In [1]:
silver_df = spark.table("bronze.raw_sales")

print("Rows:", silver_df.count())
print("Columns:", len(silver_df.columns))

silver_df.printSchema()


StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 3, Finished, Available, Finished, False)

Rows: 541909
Columns: 10
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: long (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: double (nullable = true)
 |-- Country: string (nullable = true)
 |-- SourceFile: string (nullable = true)
 |-- IngestedAt: timestamp (nullable = true)



In [2]:
from pyspark.sql.functions import col
from pyspark.sql.types import DecimalType

silver_df = (
    silver_df
    .withColumn("InvoiceNo", col("InvoiceNo").cast("string"))
    .withColumn("StockCode", col("StockCode").cast("string"))
    .withColumn("Description", col("Description").cast("string"))
    .withColumn("Quantity", col("Quantity").cast("int"))
    .withColumn("InvoiceDate", col("InvoiceDate").cast("timestamp"))
    .withColumn("UnitPrice", col("UnitPrice").cast(DecimalType(18,4)))
    .withColumn("CustomerID", col("CustomerID").cast("long").cast("string"))
    .withColumn("Country", col("Country").cast("string"))
)

silver_df.printSchema()

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 4, Finished, Available, Finished, False)

root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: decimal(18,4) (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- SourceFile: string (nullable = true)
 |-- IngestedAt: timestamp (nullable = true)



In [3]:
from pyspark.sql.functions import trim, upper

silver_df = (
    silver_df
    .withColumn("InvoiceNo", trim(col("InvoiceNo")))
    .withColumn("StockCode", trim(col("StockCode")))
    .withColumn("Description", trim(col("Description")))
    .withColumn("Country", trim(col("Country")))
)

silver_df.head()

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 5, Finished, Available, Finished, False)

Row(InvoiceNo='536365', StockCode='85123A', Description='WHITE HANGING HEART T-LIGHT HOLDER', Quantity=6, InvoiceDate=datetime.datetime(2010, 12, 1, 8, 26), UnitPrice=Decimal('2.5500'), CustomerID='17850', Country='United Kingdom', SourceFile='Online Retail.xlsx', IngestedAt=datetime.datetime(2026, 10, 30, 15, 16, 17, 724936))

In [4]:
from pyspark.sql.functions import col, lower, when

silver_df = (
    silver_df
    .withColumn(
        "IsCancellation",
        col("InvoiceNo").startswith("C")
    )
    .withColumn(
        "HasCustomerID",
        col("CustomerID").isNotNull()
    )
    .withColumn(
        "IsZeroPrice",
        col("UnitPrice") == 0
    )
    .withColumn(
        "IsAdjustment",
        lower(col("Description")) == "adjust bad debt"
    )
    .withColumn(
        "IsNegativePrice",
        col("UnitPrice") < 0
    )
    .withColumn(
        "RequiresReview",
        (
            ((col("Quantity") < 0) & (~col("InvoiceNo").startswith("C"))) |
            col("Description").isNull() |
            (lower(col("Country")) == "unspecified") |
            (col("IsNegativePrice") & (~col("IsAdjustment")))
        )
    )
)

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 6, Finished, Available, Finished, False)

In [5]:
silver_df.groupBy(
    "IsCancellation",
    "HasCustomerID",
    "IsZeroPrice",
    "IsAdjustment",
    "IsNegativePrice",
    "RequiresReview"
).count().orderBy("count", ascending=False).show(truncate=False)

print(
    "Requires Review rows: ", silver_df.filter(col("RequiresReview") == True).count()
)

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 7, Finished, Available, Finished, False)

+--------------+-------------+-----------+------------+---------------+--------------+------+
|IsCancellation|HasCustomerID|IsZeroPrice|IsAdjustment|IsNegativePrice|RequiresReview|count |
+--------------+-------------+-----------+------------+---------------+--------------+------+
|false         |true         |false      |false       |false          |false         |529657|
|true          |true         |false      |false       |false          |false         |9288  |
|false         |true         |true       |false       |false          |true          |1336  |
|false         |true         |true       |false       |false          |false         |1179  |
|false         |true         |false      |false       |false          |true          |446   |
|false         |true         |false      |true        |true           |false         |2     |
|false         |true         |false      |true        |false          |false         |1     |
+--------------+-------------+-----------+------------+-----

In [6]:
silver_df = silver_df.withColumn(
    "TransactionType",
    when(col("IsCancellation"), "Cancellation")
    .when(col("IsAdjustment"), "AccountingAdjustment")
    .when(col("RequiresReview"), "Review")
    .otherwise("Sale")
)

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 8, Finished, Available, Finished, False)

In [7]:
silver_df.groupBy("TransactionType").count().orderBy("count", ascending=False).show()

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 9, Finished, Available, Finished, False)

+--------------------+------+
|     TransactionType| count|
+--------------------+------+
|                Sale|530836|
|        Cancellation|  9288|
|              Review|  1782|
|AccountingAdjustment|     3|
+--------------------+------+



In [8]:
silver_df = silver_df.withColumn("LineAmount", (col("Quantity") * col("UnitPrice")))

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 10, Finished, Available, Finished, False)

In [9]:
silver_df.select(
    "Quantity",
    "UnitPrice",
    "LineAmount",
    "TransactionType"
).show(20, truncate=False)

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 11, Finished, Available, Finished, False)

+--------+---------+----------+---------------+
|Quantity|UnitPrice|LineAmount|TransactionType|
+--------+---------+----------+---------------+
|6       |2.9500   |17.7000   |Sale           |
|6       |2.5500   |15.3000   |Sale           |
|12      |1.4500   |17.4000   |Sale           |
|2       |7.9500   |15.9000   |Sale           |
|2       |7.9500   |15.9000   |Sale           |
|4       |5.9500   |23.8000   |Sale           |
|12      |1.2500   |15.0000   |Sale           |
|12      |1.6500   |19.8000   |Sale           |
|5       |2.9500   |14.7500   |Sale           |
|10      |1.9500   |19.5000   |Sale           |
|10      |1.9500   |19.5000   |Sale           |
|10      |1.9500   |19.5000   |Sale           |
|24      |1.2500   |30.0000   |Sale           |
|8       |1.9500   |15.6000   |Sale           |
|8       |1.9500   |15.6000   |Sale           |
|6       |2.1000   |12.6000   |Sale           |
|6       |2.1000   |12.6000   |Sale           |
|12      |0.8500   |10.2000   |Sale     

In [10]:
from pyspark.sql.functions import concat_ws, sha2, coalesce, lit

silver_df = silver_df.withColumn(
    "Country",
    when(col("Country") == "EIRE", "Ireland")
    .when(col("Country") == "RSA", "South Africa")
    .otherwise(col("Country"))
)

silver_df = silver_df.withColumn(
    "TransactionID",
    sha2(
        concat_ws(
            "||",
            coalesce(col("InvoiceNo"), lit("NULL")),
            coalesce(col("StockCode"), lit("NULL")),
            coalesce(col("Description"), lit("NULL")),
            coalesce(col("Quantity").cast("string"), lit("NULL")),
            coalesce(col("InvoiceDate").cast("string"), lit("NULL")),
            coalesce(col("UnitPrice").cast("string"), lit("NULL")),
            coalesce(col("CustomerID"), lit("NULL")),
            coalesce(col("Country"), lit("NULL"))
        ),
        256
    )
)

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 12, Finished, Available, Finished, False)

In [11]:
duplicate_ids = (
    silver_df
    .groupBy("TransactionID")
    .count()
    .filter(col("count") > 1)
)

print("Repeated TransactionIDs:", duplicate_ids.count())

duplicate_ids.orderBy("count", ascending=False).show(20, truncate=False)

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 13, Finished, Available, Finished, False)

Repeated TransactionIDs: 4879
+----------------------------------------------------------------+-----+
|TransactionID                                                   |count|
+----------------------------------------------------------------+-----+
|28fa21b9424bfbfd70423935a49687d269b4fdcf4ab27995f9286c2073adeb5f|20   |
|22220f3b525fd949bcdf2d618c14f706db5e25b0ce07b36c25e02faffebfce15|12   |
|102c70c601e42d899f71580944e2519b5cb4d032dfe38bafefe49d5f961c8292|8    |
|456b9e244d46ecb61fcbcb08dec6a8cfdc5994141e9d83ec693baae6c699f1d9|6    |
|42150ba6b597fa78b2ba088da4b175f8b35ad0e8947535190b8a397d3b2eaeb7|6    |
|568a35c2f3013867f7b8aeb4d92102a81de056fcffa6333ad7ac1d90a4cba61d|6    |
|ed0fa3ed271186cd95645df6446e431bc72f42c3f81e7f35f84e602034783bf2|6    |
|ac192e42e27395ced9e3769bdae4e9033add718c9887d4933713e29aa248feb3|6    |
|28d859d18ec8efb35b01a2110ed6240d265278cb52ec4638a153d591fc01ad9e|6    |
|6d4d408b65a68c001700ee863fabd4c9c83e77c1f287386769178d6b1f393e89|5    |
|ae408a70aecf27635936

In [12]:
from pyspark.sql.functions import sum as spark_sum

duplicate_summary = (
    duplicate_ids
    .withColumn("ExtraRows", col("count") - 1)
)

duplicate_summary.selectExpr(
    "sum(ExtraRows) as RowsRemovedIfDeduplicated"
).show()

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 14, Finished, Available, Finished, False)

+-------------------------+
|RowsRemovedIfDeduplicated|
+-------------------------+
|                     5268|
+-------------------------+



In [13]:
top_duplicate_id = (
    duplicate_ids
    .orderBy("count", ascending=False)
    .first()["TransactionID"]
)

silver_df.filter(
    col("TransactionID") == top_duplicate_id
).show(50, truncate=False)

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 15, Finished, Available, Finished, False)

+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+------------------+--------------------------+--------------+-------------+-----------+------------+---------------+--------------+---------------+----------+----------------------------------------------------------------+
|InvoiceNo|StockCode|Description                   |Quantity|InvoiceDate        |UnitPrice|CustomerID|Country       |SourceFile        |IngestedAt                |IsCancellation|HasCustomerID|IsZeroPrice|IsAdjustment|IsNegativePrice|RequiresReview|TransactionType|LineAmount|TransactionID                                                   |
+---------+---------+------------------------------+--------+-------------------+---------+----------+--------------+------------------+--------------------------+--------------+-------------+-----------+------------+---------------+--------------+---------------+----------+-------------------------------------------

In [14]:
silver_df = silver_df.dropDuplicates(["TransactionID"])

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 16, Finished, Available, Finished, False)

In [15]:
print("Rows after deduplication:", silver_df.count())

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 17, Finished, Available, Finished, False)

Rows after deduplication: 536641


In [16]:
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("silver.sales_transactions")
)

print("Silver table created successfully.")

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 18, Finished, Available, Finished, False)

Silver table created successfully.


In [17]:
silver_check = spark.table("silver.sales_transactions")

print("Rows:", silver_check.count())
print("Columns:", len(silver_check.columns))

silver_check.printSchema()

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 19, Finished, Available, Finished, False)

Rows: 536641
Columns: 19
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: timestamp (nullable = true)
 |-- UnitPrice: decimal(18,4) (nullable = true)
 |-- CustomerID: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- SourceFile: string (nullable = true)
 |-- IngestedAt: timestamp (nullable = true)
 |-- IsCancellation: boolean (nullable = true)
 |-- HasCustomerID: boolean (nullable = true)
 |-- IsZeroPrice: boolean (nullable = true)
 |-- IsAdjustment: boolean (nullable = true)
 |-- IsNegativePrice: boolean (nullable = true)
 |-- RequiresReview: boolean (nullable = true)
 |-- TransactionType: string (nullable = true)
 |-- LineAmount: decimal(29,4) (nullable = true)
 |-- TransactionID: string (nullable = true)



In [18]:
from pyspark.sql.functions import col, count, when, sum as spark_sum

# 1. Confirm row count
print("Silver rows:", silver_df.count())

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 20, Finished, Available, Finished, False)

Silver rows: 536641


In [19]:
# 2. Confirm TransactionID uniqueness
print(
    "Duplicate TransactionIDs:",
    silver_df.groupBy("TransactionID")
    .count()
    .filter(col("count") > 1)
    .count()
)

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 21, Finished, Available, Finished, False)

Duplicate TransactionIDs: 0


In [20]:
# 3. Check transaction-type totals
silver_df.groupBy("TransactionType") \
    .count() \
    .orderBy("count", ascending=False) \
    .show()

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 22, Finished, Available, Finished, False)

+--------------------+------+
|     TransactionType| count|
+--------------------+------+
|                Sale|525609|
|        Cancellation|  9251|
|              Review|  1778|
|AccountingAdjustment|     3|
+--------------------+------+



In [21]:
# 4. Check for unexpected nulls in key fields
silver_df.select(
    spark_sum(when(col("InvoiceNo").isNull(), 1).otherwise(0)).alias("NullInvoiceNo"),
    spark_sum(when(col("StockCode").isNull(), 1).otherwise(0)).alias("NullStockCode"),
    spark_sum(when(col("Quantity").isNull(), 1).otherwise(0)).alias("NullQuantity"),
    spark_sum(when(col("InvoiceDate").isNull(), 1).otherwise(0)).alias("NullInvoiceDate"),
    spark_sum(when(col("UnitPrice").isNull(), 1).otherwise(0)).alias("NullUnitPrice"),
    spark_sum(when(col("TransactionID").isNull(), 1).otherwise(0)).alias("NullTransactionID")
).show()

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 23, Finished, Available, Finished, False)

+-------------+-------------+------------+---------------+-------------+-----------------+
|NullInvoiceNo|NullStockCode|NullQuantity|NullInvoiceDate|NullUnitPrice|NullTransactionID|
+-------------+-------------+------------+---------------+-------------+-----------------+
|            0|            0|           0|              0|            0|                0|
+-------------+-------------+------------+---------------+-------------+-----------------+



In [22]:
# 5. Preview LineAmount
silver_df.select(
    "InvoiceNo",
    "Quantity",
    "UnitPrice",
    "LineAmount",
    "TransactionType"
).show(20, truncate=False)

StatementMeta(, 4751ea09-2c3d-4c1b-997b-7cacea489994, 24, Finished, Available, Finished, False)

+---------+--------+---------+----------+---------------+
|InvoiceNo|Quantity|UnitPrice|LineAmount|TransactionType|
+---------+--------+---------+----------+---------------+
|547209   |12      |1.6500   |19.8000   |Sale           |
|539471   |3       |1.9500   |5.8500    |Sale           |
|556070   |48      |0.4200   |20.1600   |Sale           |
|569965   |10      |12.7500  |127.5000  |Sale           |
|566255   |240     |0.4200   |100.8000  |Sale           |
|571082   |1       |1.2500   |1.2500    |Sale           |
|557502   |2       |10.7900  |21.5800   |Sale           |
|552702   |1       |5.7900   |5.7900    |Sale           |
|556258   |24      |0.5500   |13.2000   |Sale           |
|579141   |1       |3.7500   |3.7500    |Sale           |
|573173   |4       |3.7500   |15.0000   |Sale           |
|552655   |4       |1.4500   |5.8000    |Sale           |
|574893   |1       |1.2500   |1.2500    |Sale           |
|580367   |3       |3.2900   |9.8700    |Sale           |
|543604   |24 